In [ ]:
# I learned the approach implemented in this notebook
# https://www.kaggle.com/code/yuriygreben/birdclef-26-onnx-perch-dual-ssms-vectorized-mlp
# will try to implemented some of the ideas of the notebook.

In [ ]:
#%pip install -q --no-deps /kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

In [ ]:
mode = 'train_offline'
run_build_cache = True

In [ ]:
import librosa
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from pathlib import Path
if "offline" in mode:
    print('working offline')
    import tensorflow as tf
    import math
    import soundfile as sf
    import mlflow
    tf.config.set_visible_devices([], 'GPU')  # force CPU
elif "online" in mode or mode == 'submit':
    %pip install -q --no-deps /kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
import onnxruntime as ort

## Download perch_v2_cpu and run it and cache the results

First, try to run ssm on the audio file in train_soundscapes folder


In [ ]:
kaggle_onnx_path = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/perch_v2.onnx')
# dictionary to perch_v2 for different modes
onnx_path_dict = {
    'train_offline': Path('../models/perch_onnx/perch_v2.onnx'),
    'train_online': kaggle_onnx_path,
    'submit': kaggle_onnx_path,
    'test_submit': kaggle_onnx_path,
}
kaggle_base_path = Path('/kaggle/input/competitions/birdclef-2026')
base_path_dict = {
    'train_offline': Path('../data'),
    'train_online': kaggle_base_path,
    'submit': kaggle_base_path,
    'test_submit': kaggle_base_path,
}
onnx_path = onnx_path_dict[mode]
base_path = base_path_dict[mode]

In [ ]:
session_option = ort.SessionOptions()
session_option.intra_op_num_threads = 4
onnx_session = ort.InferenceSession(onnx_path, session_options = session_option, providers = ['CPUExecutionProvider'])
onnx_ipt_name = onnx_session.get_inputs()[0].name
print(f'Onnx input ame: {onnx_ipt_name}')
onnx_opt_map = {o.name: i for i, o in enumerate(onnx_session.get_outputs())}
print(f'Onnx output maps: {onnx_opt_map}')

### Build soundscapes cache


### Get the sites and the hours where and when sounds are recorded

Sites and the hours will be added as additional features along with the features obtained from perch-v2


In [ ]:
import re
# train_soundscapes file names has site up to 20, so this pattern is used
# if it does not match, it means the files in test_sounscapes folder may have different patterns
filename_pattern = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(\w{3})_(\w{8})_(\w{6})\.ogg")
def get_site_month_hour(filename) -> tuple[str, int,int]:
    """
    Arg:
    filename: the filename of a file in soundscape folder that has site and hour information
    Return:
    A tuple with filename as string and site as an integer
    """
    _, site, date, hour = filename_pattern.match(filename).groups()

    return (site, int(date[4:6]), int(hour[:2]))

    
def build_file_name_arr(soundscapes_folder_path):
    return 0
    
get_site_month_hour('BC2026_Train_0001_S08_20250606_030007.ogg')


In [ ]:

train_soundscapes_label_df = pd.read_csv(base_path / 'train_soundscapes_labels.csv')
train_soundscapes_label_df['site'], train_soundscapes_label_df['month'], train_soundscapes_label_df['hour'] = zip(*train_soundscapes_label_df['filename'].apply(get_site_month_hour))

sites = sorted(train_soundscapes_label_df['site'].unique())
hours = sorted(train_soundscapes_label_df['hour'].unique())
months = sorted(train_soundscapes_label_df['month'].unique())
print(f'sites: {sites} \n hours: {hours}\n months: {months}')

In [ ]:
# apparently, each row in train_soundscapes_label_df is duplicated
print(f'Length of train_soundscapes_label_df before deduplicate: {len(train_soundscapes_label_df)}')
train_soundscapes_label_df = train_soundscapes_label_df.drop_duplicates()
print(f'Length of train_soundscapes_label_df after deduplicate: {len(train_soundscapes_label_df)}')

In [ ]:
train_soundscapes_label_df.head()

In [ ]:

if "offline" in mode:
    perch_label_path = Path('../models/perch_onnx/labels.csv')
    
elif "online" in mode or mode == 'submit':
    perch_label_path = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/labels.csv')

perch_df = pd.read_csv(perch_label_path)
# rename the column to match the column name in taxonomy.csv
perch_df.rename(columns={'inat2024_fsd50k': 'scientific_name'}, inplace=True)
perch_df.head()

In [ ]:
taxonomy_df = pd.read_csv(base_path / 'taxonomy.csv')
taxonomy_df.head()

In [ ]:
taxonomy_join_perch_df = taxonomy_df.merge(perch_df.rename_axis("perch_idx").reset_index(), on='scientific_name', how='left')
taxonomy_join_perch_df.head()


In [ ]:
taxonomy_df.head()

In [ ]:

nan_mask = taxonomy_join_perch_df['perch_idx'].isna()
print(f"Number of nan in perch_idx: {nan_mask.sum()}")

In [ ]:
# fill NaN in 'perch_idx' with len(perch_df)-an unknown species
unknow_species_idx = len(perch_df)
taxonomy_join_perch_df['perch_idx'] = taxonomy_join_perch_df['perch_idx'].fillna(unknow_species_idx)
taxonomy_join_perch_df['perch_idx'] = taxonomy_join_perch_df['perch_idx'].astype(np.int32)
taxonomy_join_perch_df.head()

In [ ]:
submission_df = pd.read_csv(base_path / 'sample_submission.csv')
submission_df.head()

In [ ]:
#labels of species in the submission file
submission_species_labels = submission_df.columns[1:].to_list()
# assert that species in submision are the same and are in the same order as primary_labels in taxonomy
assert(submission_species_labels == taxonomy_df['primary_label'].to_list())
"Mismatch between labels in submission and taxonomy "
#species that are in submision but and in perch

label_to_perch_idx = taxonomy_join_perch_df.set_index("primary_label")["perch_idx"]
species_in_perch_mask = taxonomy_join_perch_df['perch_idx'] != unknow_species_idx
species_not_in_perch_mask = ~species_in_perch_mask
#scientific names of species not in perch
#positions of species in perch (also in taxaxonomy) in the taxonomy table
species_in_perch_positions = taxonomy_join_perch_df[species_in_perch_mask].index.to_list()
#positions of species not in perch (but in taxaxonomy) in the taxonomy table
species_not_in_perch_positions = taxonomy_join_perch_df[species_not_in_perch_mask].index.to_list()
print(species_in_perch_positions)
print(species_not_in_perch_positions)
# for each species not in perch, returns list of scienticfic names of all species which are in the same genus 
# as the species not in perch



In [ ]:
# rows of species that are not in perch
taxonomy_join_perch_df[species_not_in_perch_mask]

In [ ]:
# perch indices of species that are in taxonomy and in perch
perch_idx_species_in_perch = taxonomy_join_perch_df[species_in_perch_mask]['perch_idx'].to_list()
print(perch_idx_species_in_perch)

In [ ]:
# genus of species that are not in perch
not_in_perch_genera = taxonomy_join_perch_df[species_not_in_perch_mask]['scientific_name'].apply(lambda x: x.split(' ')[0]).unique()
not_in_perch_genera



In [ ]:
def get_genus_of_species(species):
    """
    Arg:
    species_not_in_perch: a string representing a species not in perch
    Return:
    a string representing the genus of the species
    """
    return species.split(' ')[0]
def get_perch_idx_of_species_of_same_genus(genus):
    """"
    Given a genus, return a list of all perch indices of species in the same genus 
    Args: 
    genus: a string representing a genus
    Return:
    a list of perch indices of species in the same genus 
    """
    genus_species = perch_df[perch_df['scientific_name'].str.contains(genus)]
    genus_species_idx = genus_species.index.tolist()
    return genus_species_idx
def get_perch_idx_of_species_of_same_genus_as_species_not_in_perch(species_not_in_perch):
    """
    Arg:
    species_not_in_perch: a string representing a species not in perch
    Return:
    a list of perch indices of species in the same genus
    """
    genus = get_genus_of_species(species_not_in_perch)
    return get_perch_idx_of_species_of_same_genus(genus)

# for each species not in perch, get the list of perch indices of species in the same genus
not_in_perch_idx_list = taxonomy_join_perch_df[species_not_in_perch_mask]['scientific_name'].apply(get_perch_idx_of_species_of_same_genus_as_species_not_in_perch)
for idx in not_in_perch_idx_list.index:
    print(f'{idx}: {not_in_perch_idx_list[idx]}')

In [ ]:
SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: int) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    Arguments:
        path: path to .ogg file
        offset_sec: offset in seconds into the file
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
sample_path = base_path / 'train_soundscapes' / 'BC2026_Train_0001_S08_20250606_030007.ogg'
chunk = load_chunk(sample_path, offset_sec=0.0)
print('Shape:', chunk.shape)   # (160000,)
print('dtype:', chunk.dtype)   # float32
print('Range:', chunk.min(), chunk.max())

In [ ]:
taxonomy_df['primary_label'].values

In [ ]:
def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = waveform[np.newaxis, :]
    outs = onnx_session.run(None, {onnx_ipt_name: inp})
    # print(len(outs))
    emb  = outs[onnx_opt_map['embedding']].astype(np.float32)
    label_logits = outs[onnx_opt_map['label']]  # (1, 508)

    
    return emb, label_logits # (1536,)

embedding, label_logtis = extract_embedding(chunk)
print(f'Embedding shape: {embedding.shape}')
print(f'Label logits shape: {label_logtis.shape}')

In [ ]:
train_soundscapes_label_df.columns

In [ ]:
# Building cache for train_soundscapes
# Only process the files in train_soundscapes_labels.csv

N_CLASSES = 234
N_WINDOWS = 12
EMBEDDING_DIMENSION = 1536
file_windows = train_soundscapes_label_df.groupby('filename').size()
full_windows_files = file_windows[file_windows == N_WINDOWS].index.to_list()
full_windows_files_mask = train_soundscapes_label_df['filename'].isin(full_windows_files)
# a sub df of train_soundscapes_label_df containing rows from 'full" 1 minute files
full_windows_files_df = (train_soundscapes_label_df[full_windows_files_mask]
                        .sort_values(['filename', 'end'])
                        .reset_index(drop=False))

filenames = (base_path / 'train_soundscapes').glob('*.ogg')
print(len(list(filenames)))
print(len(train_soundscapes_label_df))
full_windows_filenames = full_windows_files_df['filename'].unique()
train_rows_number = len(full_windows_filenames) * N_WINDOWS
def build_train_soundscapes_cache():
    
    row_ids = np.empty(train_rows_number, dtype=object)
    filenames = np.empty(train_rows_number, dtype=object)
    sites = np.empty(train_rows_number, dtype=object)
    months = np.empty(train_rows_number, dtype=object)
    hours = np.empty(train_rows_number, dtype=object)
    scores = np.empty((train_rows_number, N_CLASSES), dtype=np.float32)
    embeddings = np.empty((train_rows_number, EMBEDDING_DIMENSION), dtype=np.float32)
    stop_iteration = len(full_windows_filenames)
    for i in range(stop_iteration):
        train_soundscape_filename = full_windows_filenames[i]
        train_soundscape_path = base_path / 'train_soundscapes' / train_soundscape_filename
        site, month, hour = get_site_month_hour(train_soundscape_filename)
        for j in range(N_WINDOWS):
            off_set_sec = j*5
            chunk = load_chunk(train_soundscape_path, off_set_sec)
            embedding, label_logits = extract_embedding(chunk)
            row_ids[i*N_WINDOWS+j] = train_soundscape_filename[:-4]+'_'+str(off_set_sec+5)
            filenames[i*N_WINDOWS+j] = train_soundscape_filename
            sites[i*N_WINDOWS+j] = site
            months[i*N_WINDOWS+j] = month
            hours[i*N_WINDOWS+j] = hour
            scores[i*N_WINDOWS+j,species_in_perch_positions] = label_logits[0, perch_idx_species_in_perch]
            for taxonomy_idx in not_in_perch_idx_list.index:
                # take the max across all genus-mates per row
                # intuition: if the "max" genus-mate is present, the un-mapped species
                # is likely to present too.
                scores[i*N_WINDOWS+j, taxonomy_idx] = np.max(label_logits[0, not_in_perch_idx_list[taxonomy_idx]])
            embeddings[i*N_WINDOWS+j] = embedding
    meta_df = pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'month': months, 'hour': hours})   
    print(len(meta_df))
    META_CACHE_PATH = base_path / 'meta_train_soundscapes.parquet'
    meta_df.to_parquet(META_CACHE_PATH)
    TRAIN_SOUNDSCAPES_CACHE_PATH = base_path / "train_soundscapes_cache.npz"
    np.savez_compressed(
        TRAIN_SOUNDSCAPES_CACHE_PATH,
        embeddings=embeddings,
        scores=scores,
        primary_labels=np.array(taxonomy_df['primary_label'].values),
    )
if run_build_cache:
    build_train_soundscapes_cache()

### Dealing with labels in train_soundscapes_labels.csv


In [ ]:
def convert_labels_string_to_list(label_string):
    label_list = label_string.split(';')
    return label_list
    # return [label.strip() for label in label_list]
convert_labels_string_to_list('22967;22973;517063;trsowl')

In [ ]:

# Scores only for files that are full, i.e, 1 minute files which have 12 window
Y_full_scores = np.zeros((train_rows_number, N_CLASSES), dtype=np.uint8)
PRIMARY_LABELS = taxonomy_df['primary_label'].values
label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
for i in range(train_rows_number):
    labels = full_windows_files_df['primary_label'].iloc[i].split(';')
    label_idx_list = [label_to_idx[label] for label in labels]
    Y_full_scores[i, label_idx_list] = 1



### Build model


### Simplified Mamba Block
Slightly different from the SelectiveSSM class in the protoSSM dual vectorized mlp notebook. It is a simplified mamba block

In [ ]:
import torch.nn.functional as F
hyper_parameters = {
    'd_model': 128, # general size of matrixes that would flow between layers
    'd_state': 16, # number of states of SSM the larger the d_states the more stored hidden information
    'n_heads': 2,
    'n_fused_layers': 2
}
class SimplifiedMambaBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv_kernel=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=True)
        self.one_d_conv = nn.Conv1d(
            d_model, # in_channels - numper of input features
            d_model, # out_channels - number of output features
            kernel_size=d_conv_kernel, # how many consecutive timesteps the filter looks at once
            padding=d_conv_kernel-1, # zero-padding added to both ends of the sequence
            groups=d_model, # depthwise convolution
        )
        self.dt_project = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1) # (D,N) N=d_state, -1: keeps dimesion -1 unchange
        self.A_log = nn.Parameter(torch.log(A)) # log of A  
        self.B_project = nn.Linear(d_model, d_state, bias=False)
        self.C_project = nn.Linear(d_model, d_state, bias=False)
        self.D = nn.Parameter(torch.ones(d_model))  
        self.out_project = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B_size, T, D_size = x.shape # (B, T, D) (D= d_model) B = B_size = batch size
        x_and_res = self.in_proj(x) # (B, T, 2 * D) 
        x, res = x_and_res.chunk(2, dim=-1) # two (B, T, D)
        x = self.one_d_conv(x.transpose(1, 2))[:,:,:T].transpose(1, 2) # covolution time dimension, then transpose back
        x = F.silu(x) # (B, T, D)
        dt = self.dt_project(x) # (B, T, D)@(D,D) -> (B, T, D)
        dt = F.softplus(dt) # (B, T, D)
        A = -torch.exp(self.A_log) # (D, N) N=d_state
        B = self.B_project(x) # (B, T, N)  
        C = self.C_project(x) # (B, T, N) 
        h = torch.zeros(B_size, self.d_model, self.d_state, device=x.device) # (B, D, N)
        ys = []
        for t in range(T):
            # A (D, N) A[None] (1, D, N), dt (B, T, D) dt[:, t, :, None] (B, D, 1) 
            # A[None] * dt[:, t, :, None] broadcast -> (B, D, N)
            dA = torch.exp(A[None] * dt[:, t, :, None]) 
            # dt[:,t,:,Non] (B,D,1) * B (B, T, N) B[:,t,None,:] (B, 1, N) broadcast -> (B, D, N)
            dB = dt[:,t,:,None] * B[:,t,None,:]
            h = dA * h + dB * x[:,t,:,None] # (B, D, N) * (B, D, 1) -> (B, D, N) 
            y = (h * C[:, t, None, :]).sum(-1) # (B, D, N) * (B, 1, N) -> (B, D, N).sum(-1) -> (B,D)
            ys.append(y)
        y = torch.stack(ys, dim=1) # (B, T, D)
        y = y + (x * self.D[None, None, :]) # (B, T, D) D skip (over x)
        y = y * F.silu(res) # (B, T, D) gate with residual
        return self.out_project(y) 

### Combined Perch Mamba attention
Combining Perch prediction with a model fusing simplified mamba and attention blocks

In [ ]:
class CombinedPerchMambaAttention(nn.Module):
    def __init__(
            self, 
            emb_dim = EMBEDDING_DIMENSION, # dimesion of perch embeddings 
            d_model = 128, 
            d_state = 16,
            n_classes = N_CLASSES,
            n_windows = N_WINDOWS,
            n_sites = 20, 
            dropout = 0.15,
            meta_dim = 8, # sites, months, hours
            n_mamba_attention_blocks = 2, 
            n_cross_attention_heads = 2
        ):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.n_mamba_attention_blocks = n_mamba_attention_blocks
        self.n_cross_attention_heads = n_cross_attention_heads
        self.input_project = nn.Sequential(
            nn.Linear(emb_dim, d_model),
            nn.LayerNorm(d_model), # normalize across features
            nn.GELU(), # Gaussian Error Linear Unit smoother than RELU
            nn.Dropout(dropout)
        )
        # position encoder so that model knows which windows
        # start small with 0.02 factor
        self.windows_pos = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_embedding = nn.Embedding(n_sites, meta_dim)
        self.month_embedding = nn.Embedding(12, meta_dim)
        self.hour_embedding = nn.Embedding(24, meta_dim)
        self.meta_project = nn.Linear(meta_dim * 3, d_model)
        # nn.ModuleList just python list, but it lets python know that 
        # those are parameters of the model
        self.mamba_forward = nn.ModuleList([SimplifiedMambaBlock(d_model, d_state) for _ in range(n_mamba_attention_blocks)])
        self.mamba_backward = nn.ModuleList([SimplifiedMambaBlock(d_model, d_state) for _ in range(n_mamba_attention_blocks)])
        self.mamba_merge = nn.ModuleList([nn.Linear(d_model*2, d_model) for _ in range(n_mamba_attention_blocks)])
        self.mamba_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_mamba_attention_blocks)])
        self.dropout = nn.Dropout(dropout)  
        self.cross_attention = nn.ModuleList([
            nn.MultiheadAttention(d_model, n_cross_attention_heads, dropout=dropout, batch_first=True)
            for _ in range(n_mamba_attention_blocks)
        ])
        self.cross_attention_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_mamba_attention_blocks)])
        # baseline of perch predictions for each class
        self.perch_baseline = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        # temperature to sharpen rather flat cosine similarity of perch predictions and mamba-attention predictions
        self.temperature = nn.Parameter(torch.tensor(5.0)) # single scalar initialized to 5
        # class_bias to correct the imbalance in the bird class distributions, 
        # rare classes are boosted, common classes are suppressed
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        # a learnable parameter to blend MambaAttention with Perch logits
        # later alpha=0 → Pure MambaAttention, alpha=1 → Pure Perch
        # sigmoid(0)=0.5
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes)) # 0.0 -> 0.5 sigmoid

    def init_perch_baseline(self, perch_embeddings, y_labels):
        """
        take perch_embeddings from cache and y_labels from train soundscapes
        then porpulate the self.perch_baseline
        """
        self.eval() # disable dropout during init
        with torch.no_grad():
            # mean embedding for each class
            x = self.input_project(perch_embeddings)
            for c in range(self.n_classes):
                class_mask = y_labels[:,c] > 0.5 # boolean mask

                if class_mask.sum() > 0:
                    # mean of embeddings for this class
                    self.perch_baseline.data[c] = F.normalize(
                        x[class_mask].mean(dim=0),
                        dim=-1
                    )
        self.train() # restore training mode
    def forward(self, perch_emb, perch_logits, site_ids=None, months=None, hours=None):
        B, T, _ = perch_emb.shape
        # project embeddings to d_model space
        x = self.input_project(perch_emb) # (B, n_windows, d_model)
        # add positional encoding
        x = x + self.windows_pos[:, :T, :] # (B, n_windows, d_model)
        # add metadata embeddings
        if site_ids is not None and months is not None and hours is not None:
            meta = torch.cat([
                self.site_embedding(site_ids),
                self.month_embedding(months),
                self.hour_embedding(hours)
            ], dim=-1) # (B, meta_dim * 3)
            meta = self.meta_project(meta) # (B, d_model)
            x = x + meta.unsqueeze(1) # (B, 1, d_model)

        # apply mamba blocks
        for i in range(self.n_mamba_attention_blocks):
            res = x
            # forward and backward mamba
            x_f = self.mamba_forward[i](x)
            # (B, T , D), flip(1) -> flip the window dimension
            # original: [w0, w1, w2, ..., w10, w11]
            # first flip: [w11, w10, w9, ..., w1, w0]
            # second flip: flip the output of bwd(h.flip(1)) back to the original order
            x_b = self.mamba_backward[i](torch.flip(x, dims=[1]))
            x_b = torch.flip(x_b, dims=[1])
            # combine forward and backward
            x = self.mamba_merge[i](torch.cat([x_f, x_b], dim=-1)) # (B, T, 2 * D) -> (B, T, D)
            x = self.dropout(x)
            x = self.mamba_norm[i](x + res)
            # cross attention
            x_attn, _ = self.cross_attention[i](x, x, x)
            x = self.cross_attention_norm[i](x + x_attn)
        
         # just L2 normalization (cosine similarity)
        # x_n(b,t) = x[b,t]/||x[b,t]||2
        # x (B, T, D) -> (B, T, 128)
        x_n = F.normalize(x, dim=-1) # normalize x
        # normalize perch_baselin
        p_n = F.normalize(self.perch_baseline, dim=-1) # (n_classes, d_model)
        

        # compute similarity between x and perch_baseline
        # x (B, n_windows, d_model) @ (n_classes, d_model).T -> (B, n_windows, n_classes)
        # multiply by F.softplus(temperature) to sharpen the similarities
        cos_sim = (
            torch.matmul(x_n, p_n.T) * F.softplus(self.temperature)
            + self.class_bias[None, None, :]
        ) # (B, n_windows, n_classes)

        # self.fusion_alpha (n_classes,) -> (1, 1, n_classes) initialized to be 0
        # so that torch.sigmoid(self.fusion_alpha[None, None, :]) = 0.5
        # if alpha = 0, trust perch_logits, if alpha = 1, trust proto, if alpha = 0.5, trust both equally
        
        if perch_logits is not None:
            # (B, n_windows, n_classes) * (1, 1, n_classes) -> (B, n_windows, n_classes)
            # (B, n_windows, n_classes) * (1, 1, n_classes) -> (B, n_windows, n_classes)
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :] # (1, 1, n_classes)
            out = cos_sim * alpha + perch_logits * (1 - alpha) # (B, n_windows, n_classes) 
        else:
            out = cos_sim 
        return out
    
    def get_parameters_number(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

